# 5.4 Time Series Anomaly Detection

這份 Notebook 使用 LSTM autoencoder 學習正常時間序列，並用重建誤差偵測異常。


## 1. 載入套件


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

tf.keras.utils.set_random_seed(42)


## 2. 建立正常與異常序列


In [ ]:
TIMESTEPS = 60

def make_normal(rng, n_samples):
    t = np.linspace(0, 2 * np.pi, TIMESTEPS)
    X = []
    for _ in range(n_samples):
        phase = rng.uniform(-0.4, 0.4)
        amp = rng.uniform(0.8, 1.2)
        signal = amp * np.sin(t + phase) + rng.normal(0, 0.06, TIMESTEPS)
        X.append(signal)
    return np.array(X, dtype='float32')[..., np.newaxis]

def make_anomaly(rng, n_samples):
    X = make_normal(rng, n_samples)
    labels = np.ones(n_samples, dtype='int64')
    for i in range(n_samples):
        if i % 2 == 0:
            pos = rng.integers(12, 48)
            X[i, pos:pos + 3, 0] += rng.uniform(1.8, 2.4)
        else:
            pos = rng.integers(25, 42)
            X[i, pos:, 0] += rng.uniform(1.0, 1.5)
    return X.astype('float32'), labels

rng = np.random.default_rng(42)
x_train = make_normal(rng, 500)
x_valid = make_normal(rng, 120)
x_test_normal = make_normal(rng, 160)
x_test_anomaly, y_anomaly = make_anomaly(rng, 160)
x_test = np.concatenate([x_test_normal, x_test_anomaly], axis=0)
y_test = np.concatenate([np.zeros(len(x_test_normal), dtype='int64'), y_anomaly], axis=0)
idx = rng.permutation(len(y_test))
x_test, y_test = x_test[idx], y_test[idx]

mean = x_train.mean(axis=(0, 1), keepdims=True)
std = x_train.std(axis=(0, 1), keepdims=True) + 1e-8
x_train_s = (x_train - mean) / std
x_valid_s = (x_valid - mean) / std
x_test_s = (x_test - mean) / std
print(x_train_s.shape, x_valid_s.shape, x_test_s.shape)


## 3. 觀察正常與異常序列


In [ ]:
plt.figure(figsize=(9, 3))
plt.plot(x_test[y_test == 0][0, :, 0], label='normal')
plt.plot(x_test[y_test == 1][0, :, 0], label='anomaly')
plt.title('Normal vs anomaly')
plt.legend()
plt.show()


## 4. 建立 LSTM autoencoder


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(TIMESTEPS, 1)),
    tf.keras.layers.LSTM(32, return_sequences=False),
    tf.keras.layers.RepeatVector(TIMESTEPS),
    tf.keras.layers.LSTM(32, return_sequences=True),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(1)),
])
model.compile(optimizer='adam', loss='mae')
model.summary()


## 5. 只用正常資料訓練


In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
history = model.fit(
    x_train_s, x_train_s,
    validation_data=(x_valid_s, x_valid_s),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0,
)
pd.DataFrame(history.history).plot(figsize=(8, 4), title='Autoencoder loss')
plt.show()


## 6. 用重建誤差設定 threshold


In [ ]:
train_recon = model.predict(x_train_s, verbose=0)
train_error = np.mean(np.abs(train_recon - x_train_s), axis=(1, 2))
threshold = np.percentile(train_error, 99)
print('threshold:', threshold)

plt.figure(figsize=(8, 3))
plt.hist(train_error, bins=30)
plt.axvline(threshold, color='red', linestyle='--', label='threshold')
plt.title('Train reconstruction error')
plt.legend()
plt.show()


## 7. 偵測測試資料異常


In [ ]:
test_recon = model.predict(x_test_s, verbose=0)
test_error = np.mean(np.abs(test_recon - x_test_s), axis=(1, 2))
y_pred = (test_error > threshold).astype('int64')
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['normal', 'anomaly']))


## 8. 視覺化單筆異常重建


In [ ]:
anomaly_idx = np.where((y_test == 1) & (y_pred == 1))[0][0]
plt.figure(figsize=(9, 3))
plt.plot(x_test_s[anomaly_idx, :, 0], label='input')
plt.plot(test_recon[anomaly_idx, :, 0], label='reconstructed')
plt.title(f'reconstruction error={test_error[anomaly_idx]:.3f}')
plt.legend()
plt.show()


## 9. 如何套用自己的資料

先定義一個 window，例如每 60 秒或每 144 筆是一個樣本。若能取得可信的正常資料，先只用正常資料訓練 autoencoder，再用重建誤差分位數設定 threshold。


## 10. 小結

Autoencoder anomaly detection 的核心是學正常 pattern，再用 reconstruction error 量化偏離程度。Threshold 需要依實務誤報率與漏報風險調整。
